# Size scaling — point cloud

Write/read/disk-size of point clouds at increasing `N`. Same
`chunk_shape` across runs so the only variable is vertex count.

Runtime: a few minutes on a laptop (the 1M case dominates).

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points

SIZES = [1_000, 10_000, 100_000, 1_000_000]
CHUNK = (200.0, 200.0, 200.0)
BIN   = (50.0, 50.0, 50.0)
SEED  = 0

## 2 · Run the sweep

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for n in SIZES:
    positions = rng.uniform(0, 1000, (n, 3)).astype(np.float32)
    intensity = rng.uniform(0, 1, n).astype(np.float32)

    store = _new_store(f'size_{n}')
    t_write, _ = _time(
        write_points, store, positions,
        chunk_shape=CHUNK, bin_shape=BIN,
        attributes={'intensity': intensity},
    )
    t_read, _ = _time(read_points, store, attribute_names=['intensity'])
    rows.append({
        'N': n,
        'write_s': round(t_write, 3),
        'read_s':  round(t_read,  3),
        'size_MB': round(_store_bytes(store) / 1e6, 2),
    })
    shutil.rmtree(Path(store).parent, ignore_errors=True)

df = pd.DataFrame(rows)

## 3 · Results

In [ ]:
df

## 4 · Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(df['N'], df['write_s'], 'o-', label='write (s)')
ax.loglog(df['N'], df['read_s'],  's-', label='read (s)')
ax.loglog(df['N'], df['size_MB'], '^-', label='size (MB)')
ax.set_xlabel('N (vertices)')
ax.set_title('Point cloud: write/read time + disk footprint vs N')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()